In [1]:
import numpy as np
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import os

# ==========================================
# PHẦN 1: THUẬT TOÁN CHOLESKY XUẤT MARKDOWN
# ==========================================

def matrix_to_latex_core(np_mat, precision=4):
    """Hàm lõi tạo chuỗi bmatrix thuần (không chứa ký hiệu $$)"""
    mat = np.copy(np_mat).astype(float)
    mat[np.abs(mat) < 1e-10] = 0.0 # Xóa lỗi -0.0
    mat = np.round(mat, precision)
    
    latex_str = [r"\begin{bmatrix}"]
    for row in mat:
        row_str = " & ".join([f"{val:g}" for val in row])
        latex_str.append("  " + row_str + r" \\")
    latex_str.append(r"\end{bmatrix}")
    return "\n".join(latex_str)

def matrix_to_latex(np_mat, precision=4):
    """Hàm bọc $$ cho ma trận đứng độc lập"""
    return "$$\n" + matrix_to_latex_core(np_mat, precision) + "\n$$"

def run_cholesky(aug_mat, n_cols_b):
    m, total_cols = aug_mat.shape
    n_cols_a = total_cols - n_cols_b
    
    A = aug_mat[:, :n_cols_a].astype(float)
    B = aug_mat[:, n_cols_a:].astype(float)
    
    md = []
    md.append("# TRÌNH BÀY LỜI GIẢI BẰNG PHƯƠNG PHÁP CHOLESKY\n")
    
    # 1. Kiểm tra điều kiện ma trận vuông
    if m != n_cols_a:
        md.append("### Lỗi: Ma trận không hợp lệ\n")
        md.append("Ma trận hệ số $A$ không phải là ma trận vuông. Không thể áp dụng phương pháp Cholesky.\n")
        return "\n".join(md)
        
    md.append("### 1. Ma trận hệ số $A$ và vế phải $B$:\n")
    md.append("**Ma trận $A$:**\n" + matrix_to_latex(A) + "\n")
    md.append("**Ma trận $B$:**\n" + matrix_to_latex(B) + "\n")

    # 2. Kiểm tra tính đối xứng
    if not np.allclose(A, A.T, atol=1e-10):
        md.append("### Lỗi: Ma trận không đối xứng\n")
        md.append("Ma trận $A$ không đối xứng ($A \\neq A^T$). Không thể áp dụng phương pháp Cholesky.\n")
        return "\n".join(md)

    n = m
    U = np.zeros((n, n))
    
    # 3. Phân tách A = U^T * U
    for i in range(n):
        # Tính đường chéo chính U[i, i]
        sum_sq = sum(U[k, i]**2 for k in range(i))
        val = A[i, i] - sum_sq
        
        # Kiểm tra tính xác định dương (giá trị dưới căn phải > 0)
        if val <= 1e-10:
            md.append("### Lỗi: Ma trận không xác định dương\n")
            md.append(f"Quá trình tính toán dừng lại tại $u_{{{i+1},{i+1}}}$ vì giá trị dưới căn $\\le 0$ (Cụ thể: `{val:g}`). Ma trận không thỏa mãn điều kiện áp dụng.\n")
            return "\n".join(md)
            
        U[i, i] = np.sqrt(val)
        
        # Tính các phần tử còn lại trên hàng i (j > i)
        for j in range(i + 1, n):
            sum_cross = sum(U[k, i] * U[k, j] for k in range(i))
            U[i, j] = (A[i, j] - sum_cross) / U[i, i]

    UT = U.T
    md.append("### 2. Phân tách $A = U^T U$:\n")
    md.append("Từ công thức Cholesky, ta tìm được **ma trận tam giác trên $U$**:\n")
    md.append(matrix_to_latex(U) + "\n")
    md.append("Suy ra **ma trận tam giác dưới $U^T$**:\n")
    md.append(matrix_to_latex(UT) + "\n")

    # 4. Giải U^T * Y = B (Forward substitution)
    Y = np.zeros_like(B)
    for i in range(n):
        for col in range(n_cols_b):
            sum_y = sum(UT[i, j] * Y[j, col] for j in range(i))
            Y[i, col] = (B[i, col] - sum_y) / UT[i, i]
            
    md.append("### 3. Giải hệ $U^T Y = B$ (Thế tiến):\n")
    md.append("Ta thu được ma trận trung gian $Y = $:\n")
    md.append(matrix_to_latex(Y) + "\n")

    # 5. Giải U * X = Y (Backward substitution)
    X = np.zeros_like(B)
    for i in range(n - 1, -1, -1):
        for col in range(n_cols_b):
            sum_x = sum(U[i, j] * X[j, col] for j in range(i + 1, n))
            X[i, col] = (Y[i, col] - sum_x) / U[i, i]

    md.append("### 4. Giải hệ $U X = Y$ (Thế ngược):\n")
    md.append("Ta thu được ma trận nghiệm $X = $:\n")
    md.append(matrix_to_latex(X) + "\n")

    md.append("### 5. Kết luận\n")
    md.append("Nghiệm duy nhất của hệ phương trình là:\n")
    md.append(matrix_to_latex(X) + "\n")

    return "\n".join(md)


# ==========================================
# PHẦN 2: GIAO DIỆN TKINTER NHẬP TỪ FILE
# ==========================================

class MatrixApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Công cụ Giải Cholesky (Hỗ trợ File txt)")
        self.root.geometry("600x500")
        
        config_frame = ttk.LabelFrame(root, text="1. Cấu hình vế phải", padding=10)
        config_frame.pack(fill="x", padx=10, pady=5)
        
        ttk.Label(config_frame, text="Số cột của ma trận B (vế phải):").grid(row=0, column=0, padx=5)
        self.cols_b_var = tk.IntVar(value=1)
        ttk.Entry(config_frame, textvariable=self.cols_b_var, width=5).grid(row=0, column=1)
        
        input_frame = ttk.LabelFrame(root, text="2. Nhập ma trận [A | B]", padding=10)
        input_frame.pack(fill="both", expand=True, padx=10, pady=5)
        
        toolbar = ttk.Frame(input_frame)
        toolbar.pack(fill="x", pady=(0, 5))
        
        ttk.Button(toolbar, text="Mở file Notepad (.txt)", command=self.load_file).pack(side="left")
        ttk.Label(toolbar, text=" Hoặc copy/paste thẳng vào ô bên dưới:", foreground="gray").pack(side="left", padx=10)
        
        self.text_area = tk.Text(input_frame, wrap="none", font=("Consolas", 11))
        self.text_area.pack(fill="both", expand=True)
        
        # Sửa sample data thành một ma trận thỏa mãn điều kiện Cholesky
        sample_data = "4 2 -2 -2\n2 10 5 2\n-2 5 6 5"
        self.text_area.insert("1.0", sample_data)
        
        btn_frame = ttk.Frame(root, padding=10)
        btn_frame.pack(fill="x")
        ttk.Button(btn_frame, text="GIẢI & XUẤT FILE MARKDOWN", command=self.solve, style="Accent.TButton").pack(pady=10)

    def load_file(self):
        filepath = filedialog.askopenfilename(filetypes=[("Text Files", "*.txt"), ("All Files", "*.*")])
        if filepath:
            with open(filepath, 'r') as f:
                content = f.read()
            self.text_area.delete("1.0", tk.END)
            self.text_area.insert(tk.END, content)

    def solve(self):
        try:
            n_cols_b = self.cols_b_var.get()
            if n_cols_b <= 0:
                raise ValueError
        except:
            messagebox.showerror("Lỗi", "Số cột của B phải là số nguyên > 0!")
            return
            
        raw_text = self.text_area.get("1.0", tk.END).strip()
        if not raw_text:
            messagebox.showerror("Lỗi", "Dữ liệu ma trận đang trống!")
            return
            
        try:
            import io
            aug_mat = np.loadtxt(io.StringIO(raw_text))
            if aug_mat.ndim == 1: 
                aug_mat = aug_mat.reshape(1, -1)
        except Exception as e:
            messagebox.showerror("Lỗi Cú Pháp", "Không thể đọc ma trận. Hãy đảm bảo dữ liệu chỉ chứa số và được phân cách bằng dấu cách hoặc tab.\nChi tiết: " + str(e))
            return
            
        m, total_cols = aug_mat.shape
        if total_cols <= n_cols_b:
            messagebox.showerror("Lỗi Kích Thước", f"Tổng số cột đọc được ({total_cols}) phải lớn hơn số cột của B ({n_cols_b})!")
            return
            
        save_path = filedialog.asksaveasfilename(
            defaultextension=".md",
            filetypes=[("Markdown Files", "*.md")],
            title="Lưu lời giải vào file"
        )
        
        if not save_path:
            return 
            
        print("\n[HỆ THỐNG] Đang giải và tạo file Markdown...")
        md_content = run_cholesky(aug_mat, n_cols_b)
        
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(md_content)
            
        messagebox.showinfo("Thành công", f"Đã xuất lời giải Cholesky siêu đẹp ra file:\n{save_path}")

if __name__ == "__main__":
    root = tk.Tk()
    style = ttk.Style()
    style.configure("Accent.TButton", font=("Arial", 10, "bold"), foreground="black")
    app = MatrixApp(root)
    root.mainloop()